table = {"area": [900, 1500, 2200], "rooms": [2, 3, 4], "price": [240, 355, 512]}
features: ['area', 'rooms']
label: price
task: regression

Predict price using features and labels

In [ ]:
mport pandas as pd

# dataset
##Area and Rooms are weight on which price changes
table = {"area": [900, 1500, 2200],
         "rooms": [2, 3, 4],
         "price": [240, 355, 512]}

df = pd.DataFrame(table)
print(df)

In [ ]:
from sklearn.linear_model import LinearRegression

# features and label
X = df[['area', 'rooms']]
y = df['price']

# train model
model = LinearRegression()
model.fit(X, y)

# coefficients
print("Coefficients:", model.coef_)
print("Intercept:", model.intercept_)

In [ ]:
# predict for a new house
new_house = [[800, 2]]  # area=1800, rooms=3
predicted_price = model.predict(new_house)
print("Predicted Price:", predicted_price[0])

Problem 2: Three-way split and model selection
Load load_wine (from sklearn.datasets). Split off 20 percent as test with random_state=0 and stratify=y, then split the rest into train and validation with test_size=0.25, random_state=0, and stratification. Scale the features (fit the scaler on the train part only). Train two candidates, LogisticRegression(max_iter=5000) and DecisionTreeClassifier(max_depth=2, random_state=0), pick the one with the higher validation accuracy, and print the sizes, both validation accuracies, and the chosen model's test accuracy (two decimals).

sizes train/val/test: 106 36 36
validation logreg: 1.0
validation tree: 0.86
chosen: logreg
test accuracy: 1.0

In [ ]:
##Step 1: Load and Split Data
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load dataset
X, y = load_wine(return_X_y=True)

# First split: test set (20%)
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=0, stratify=y
)

# Second split: validation set (25% of trainval)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=0, stratify=y_trainval
)

print("Train size:", X_train.shape[0])
print("Validation size:", X_val.shape[0])
print("Test size:", X_test.shape[0])

Train size: 106
Validation size: 36
Test size: 36


In [ ]:
#Step 2: Scale Features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)


In [ ]:
#Step 3: Train Candidate (different) Models
# Logistic Regression
log_reg = LogisticRegression(max_iter=5000)
log_reg.fit(X_train_scaled, y_train)
val_acc_log = accuracy_score(y_val, log_reg.predict(X_val_scaled))

# Decision Tree
tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X_train_scaled, y_train)
val_acc_tree = accuracy_score(y_val, tree.predict(X_val_scaled))

print("Validation Accuracy (LogReg):", val_acc_log)
print("Validation Accuracy (Tree):", val_acc_tree)


Validation Accuracy (LogReg): 1.0
Validation Accuracy (Tree): 0.8611111111111112


In [ ]:
#Step 4: Pick Best Model & Evaluate on Test

#https://scikit-learn.org/stable/api/sklearn.model_selection.html

if val_acc_log >= val_acc_tree:
    chosen_model = log_reg
    chosen_name = "Logistic Regression"
else:
    chosen_model = tree
    chosen_name = "Decision Tree"

test_acc = accuracy_score(y_test, chosen_model.predict(X_test_scaled))

print(f"Chosen Model: {chosen_name}")
print(f"Test Accuracy: {test_acc:.2f}")


Chosen Model: Logistic Regression
Test Accuracy: 1.00


Problem 3: Name the task type
scenarios = [
    "predict tomorrow's temperature in Celsius",
    "decide if an email is spam or not",
    "identify the handwritten digit 0-9",
    "predict a house's sale price",
    "sort a news article into one of 12 topics",
]
For each scenario print "regression", "binary classification", or "multiclass classification", in order.

['regression', 'binary classification', 'multiclass classification', 'regression', 'multiclass classification']

In [6]:
scenarios = [
    "predict tomorrow's temperature in Celsius",   # regression
    "decide if an email is spam or not",           # binary classification
    "identify the handwritten digit 0-9",          # multiclass classification
    "predict a house's sale price",                # regression
    "sort a news article into one of 12 topics",   # multiclass classification
]

task_types = [
    "regression",
    "binary classification",
    "multiclass classification",
    "regression",
    "multiclass classification"
]

#print(task_types)
for s, t in zip(scenarios, task_types):
    print(f"{s} → {t}")


predict tomorrow's temperature in Celsius → regression
decide if an email is spam or not → binary classification
identify the handwritten digit 0-9 → multiclass classification
predict a house's sale price → regression
sort a news article into one of 12 topics → multiclass classification


Problem 4: Diagnose the gap
train_acc, test_acc = 0.99, 0.63
Print whether this is overfitting or underfitting, and which number honestly reflects real-world performance.

diagnosis: overfitting
honest score: 0.63

In [10]:
train_acc, test_acc = 0.99, 0.63

if train_acc > 0.95 and test_acc < 0.75:
    diagnosis = "overfitting"
    honest_score = test_acc
else:
    diagnosis = "underfitting"
    honest_score = train_acc

print("diagnosis:", diagnosis)
print("honest score:", honest_score)


diagnosis: overfitting
honest score: 0.63


Problem 5: Spot the leakage
A teammate scales the whole dataset and then splits it into train and test. Print one line saying what leaks, and one line with the fix.

leak: the scaler is fit on all rows, so test statistics enter training
fix: split first, then fit the scaler on the training rows only

In [11]:
# Problem 5: Spot the leakage

leak = "the scaler is fit on all rows, so test statistics enter training"
fix  = "split first, then fit the scaler on the training rows only"

print("leak:", leak)
print("fix:", fix)


leak: the scaler is fit on all rows, so test statistics enter training
fix: split first, then fit the scaler on the training rows only


Problem 6: Precision and recall on imbalanced labels
Here 1 marks the rare positive class (say, fraud), and the model always guesses the majority 0.

labels = [0] * 18 + [1, 1]
preds  = [0] * 20
Compute accuracy, precision, and recall by hand (no scikit-learn), treating 1 as positive. If there are no positive predictions, report precision as 0.0. Print the three numbers, then one line on why accuracy flatters this model.

accuracy: 0.9
precision: 0.0
recall: 0.0
note: accuracy is high, but recall 0.0 means it caught none of the rare positives

In [12]:
labels = [0] * 18 + [1, 1]   # 18 zeros, 2 ones
preds  = [0] * 20            # model always predicts 0

# Accuracy = correct / total
correct = sum([l == p for l, p in zip(labels, preds)])
accuracy = correct / len(labels)

# Precision = TP / (TP + FP)
# Here TP = 0, FP = 0 → no positive predictions → precision = 0.0
precision = 0.0

# Recall = TP / (TP + FN)
# TP = 0, FN = 2 → recall = 0.0
recall = 0.0

print("accuracy:", accuracy)
print("precision:", precision)
print("recall:", recall)
print("note: accuracy is high, but recall 0.0 means it caught none of the rare positives")


accuracy: 0.9
precision: 0.0
recall: 0.0
note: accuracy is high, but recall 0.0 means it caught none of the rare positives
